In [ ]:
".\processed\presidency_segments_emfd_projection_pca.csv"

In [1]:
# export_existing_pca_info_from_scored_csv.py

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


CSV_PATH = Path(".\processed\presidency_segments_emfd_projection_pca.csv")

OUT_DIR = Path("pca_export")
OUT_DIR.mkdir(exist_ok=True)

MFD_COLS = [
    "care_p",
    "fairness_p",
    "loyalty_p",
    "authority_p",
    "sanctity_p",
]

SEG_PC_COLS = [
    "seg_PC1",
    "seg_PC2",
    "seg_PC3",
    "seg_PC4",
    "seg_PC5",
]

USECOLS = MFD_COLS + SEG_PC_COLS


# 1. Sadece gerekli kolonları oku
df = pd.read_csv(CSV_PATH, usecols=USECOLS)

X = df[MFD_COLS].fillna(0).copy()

# 2. Scaler yeniden fit et
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# 3. PCA yeniden fit et
pca = PCA(n_components=5)
X_pca_new = pca.fit_transform(X_std)

# 4. Mevcut seg_PC skorlarıyla işaret yönünü hizala
# PCA bileşenlerinin işareti keyfidir. Aynı eksen ters işaretle çıkabilir.
existing_scores = df[SEG_PC_COLS].to_numpy(dtype=float)

components = pca.components_.copy()
scores = X_pca_new.copy()

sign_rows = []

for i in range(5):
    corr = np.corrcoef(scores[:, i], existing_scores[:, i])[0, 1]

    sign = 1
    if corr < 0:
        sign = -1
        scores[:, i] *= -1
        components[i, :] *= -1

    sign_rows.append({
        "component": f"PC{i+1}",
        "corr_with_existing_seg_PC": corr,
        "sign_applied": sign,
    })

sign_df = pd.DataFrame(sign_rows)


# 5. Export tabloları
scaler_df = pd.DataFrame({
    "feature": MFD_COLS,
    "mean": scaler.mean_,
    "scale": scaler.scale_,
    "var": scaler.var_,
})

components_df = pd.DataFrame(
    components.T,
    index=MFD_COLS,
    columns=[f"PC{i+1}" for i in range(5)]
).reset_index().rename(columns={"index": "feature"})

variance_df = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(5)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "explained_variance": pca.explained_variance_,
})

# 6. Kontrol: yeniden hesaplanan score ile mevcut score korelasyonu
score_check_rows = []

for i in range(5):
    corr_after = np.corrcoef(scores[:, i], existing_scores[:, i])[0, 1]
    mae = np.mean(np.abs(scores[:, i] - existing_scores[:, i]))

    score_check_rows.append({
        "component": f"PC{i+1}",
        "corr_after_sign_alignment": corr_after,
        "mae_vs_existing_seg_PC": mae,
    })

score_check_df = pd.DataFrame(score_check_rows)


# 7. Dosyaları yaz
scaler_df.to_csv(OUT_DIR / "pca_scaler_stats.csv", index=False, encoding="utf-8-sig")
components_df.to_csv(OUT_DIR / "pca_loading_matrix_aligned.csv", index=False, encoding="utf-8-sig")
variance_df.to_csv(OUT_DIR / "pca_explained_variance.csv", index=False, encoding="utf-8-sig")
sign_df.to_csv(OUT_DIR / "pca_sign_alignment.csv", index=False, encoding="utf-8-sig")
score_check_df.to_csv(OUT_DIR / "pca_score_reconstruction_check.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUT_DIR / "pca_export_summary.xlsx", engine="openpyxl") as writer:
    scaler_df.to_excel(writer, sheet_name="scaler_stats", index=False)
    components_df.to_excel(writer, sheet_name="loading_matrix_aligned", index=False)
    variance_df.to_excel(writer, sheet_name="explained_variance", index=False)
    sign_df.to_excel(writer, sheet_name="sign_alignment", index=False)
    score_check_df.to_excel(writer, sheet_name="score_check", index=False)

export_json = {
    "mfd_cols": MFD_COLS,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "scaler_var": scaler.var_.tolist(),
    "pca_components_aligned": components.tolist(),
    "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    "explained_variance": pca.explained_variance_.tolist(),
    "sign_alignment": sign_rows,
    "score_check": score_check_rows,
}

with open(OUT_DIR / "pca_export.json", "w", encoding="utf-8") as f:
    json.dump(export_json, f, indent=2, ensure_ascii=False)


print("\nSCALER")
print(scaler_df)

print("\nLOADINGS ALIGNED")
print(components_df)

print("\nEXPLAINED VARIANCE")
print(variance_df)

print("\nSIGN ALIGNMENT")
print(sign_df)

print("\nSCORE CHECK")
print(score_check_df)

print("\nDONE")
print("Export folder:", OUT_DIR.resolve())


SCALER
       feature      mean     scale       var
0       care_p  0.042293  0.017351  0.000301
1   fairness_p  0.034381  0.014371  0.000207
2    loyalty_p  0.024461  0.011572  0.000134
3  authority_p  0.030209  0.014966  0.000224
4   sanctity_p  0.009624  0.007584  0.000058

LOADINGS ALIGNED
       feature       PC1       PC2       PC3       PC4       PC5
0       care_p -0.534473  0.315474 -0.321408 -0.521941  0.488967
1   fairness_p  0.169598  0.655387  0.613600  0.184088  0.362370
2    loyalty_p -0.380733 -0.632922  0.453501  0.168023  0.469634
3  authority_p  0.683103 -0.198672 -0.331255 -0.035922  0.618772
4   sanctity_p -0.272005  0.175736 -0.452548  0.814964  0.161752

EXPLAINED VARIANCE
  component  explained_variance_ratio  explained_variance
0       PC1                  0.321527            1.607641
1       PC2                  0.241671            1.208358
2       PC3                  0.196763            0.983816
3       PC4                  0.191489            0.957448
4   